# 17 Local Volatility Extension Optional

A cautious simplified local-volatility proxy is explored for research discussion. This is not presented as a full Dupire-calibrated production local-volatility model.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
chain = generate_synthetic_indian_market(config)
iv_data = normalize_option_chain_columns(chain)
iv_data = iv_data[(iv_data["symbol"] == "NIFTY") & (iv_data["option_type"] == "call")].dropna(subset=["strike", "maturity", "implied_volatility", "underlying_price"]).copy()
iv_data["moneyness"] = iv_data["strike"] / iv_data["underlying_price"]
pivot = iv_data.pivot_table(index="maturity", columns="moneyness", values="implied_volatility", aggfunc="mean").sort_index(axis=0).sort_index(axis=1)
pivot = pivot.interpolate(axis=0).interpolate(axis=1).ffill().bfill()
# Simplified proxy: adjust implied volatility by smooth maturity and moneyness gradients. This is a diagnostic surface, not a full Dupire result.
tau_vals = pivot.index.values.astype(float)
mon_vals = pivot.columns.values.astype(float)
iv_grid = pivot.values
if iv_grid.shape[0] > 1 and iv_grid.shape[1] > 1:
    grad_tau, grad_mon = np.gradient(iv_grid, tau_vals, mon_vals, edge_order=1)
else:
    grad_tau = np.zeros_like(iv_grid); grad_mon = np.zeros_like(iv_grid)
local_vol_proxy = np.clip(iv_grid + 0.25 * tau_vals[:, None] * grad_tau + 0.10 * (mon_vals[None, :] - 1.0) * grad_mon, 0.05, 0.60)
local_df = pd.DataFrame(local_vol_proxy, index=pivot.index, columns=pivot.columns)
local_df.to_csv(TABLES_DIR / "17_simplified_local_volatility_proxy.csv")
fig = plt.figure(figsize=(9, 6))
M, Tau = np.meshgrid(mon_vals, tau_vals * 365)
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(M, Tau, local_vol_proxy, cmap="plasma")
ax.set_title("Simplified local-volatility proxy surface")
ax.set_xlabel("Strike / spot")
ax.set_ylabel("Days")
ax.set_zlabel("Local vol proxy")
save_current_figure("17_simplified_local_volatility_proxy.png")
notes = {"status": "simplified diagnostic only", "warning": "Not a full Dupire calibration; use for research discussion and sensitivity only."}
save_output(notes, "17_local_volatility_notes.json")
local_df.head()
